In [ ]:
import requests
import pandas as pd

url = "http://apis.data.go.kr/B551011/GoCamping/basedList"

service_key = "Wnus4QpirWGI56CfvzMWDIDHMRL/mEF/qTl9gwVNbRggLYTGPFIdwBy0L51B+27d5QRbLanNmIAxPwNvl7dKPA=="
data=[]
page=1
while True:
    my_params={
        "serviceKey":service_key,
        "numOfRows":100,
        "pageNo":page,
        "MobileOS":"ETC",
        "MobileApp":"CampingMap",
        "_type":"json"
    }
    response = requests.get(url,params=my_params,timeout=30)
    response.raise_for_status()
    result = response.json()
    body = result["response"]["body"]
    total_count = int(body["totalCount"])
    items = body.get("items") or {}
    rows = items.get("item",[])
    data.extend(rows)
    if len(data) >= total_count:
        break
    page += 1

# while문을 빠져나온다음에 excel로 저장
df = pd.DataFrame(data)
df.to_excel("go_camping.xlsx", index=False)



In [10]:

import pandas as pd
import folium
from folium.plugins import MarkerCluster

df = pd.read_excel("go_camping.xlsx")
print("전체 캠핑장 숫자 : ", len(df))
display(df)

df["mapX"] = pd.to_numeric(df["mapX"], errors="coerce")
df["mapY"] = pd.to_numeric(df["mapY"], errors="coerce")
df = df.dropna(subset=["mapX", "mapY"])

m = folium.Map(location=[36.5, 127.8], zoom_start=7)
cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():

    # 예약 주소가 있을 때만 버튼 생성
    resve_url = row["resveUrl"]

    reserve_html = ""

    if pd.notna(resve_url) and str(resve_url).strip():
        reserve_html = f"""
        <a href="{resve_url}" target="_blank"
        style="background-color:blue;display:block;
        border-radius:10px;color:#fff;padding:10px 0;
        text-align:center;text-decoration:none;">
            예약하기
        </a>
        """

    popup_html = f"""
    <div style="width:300px;">
        <img src="{row['firstImageUrl']}"
        style="width:100%;height:150px;object-fit:cover;
        border-radius:10px">

        <h3>{row["facltNm"]}</h3>
        <p>주소: {row["addr1"]}</p>
        <p>전화번호: {row["tel"]}</p>

        {reserve_html}
    </div>
    """

    folium.Marker(
        location=[row["mapY"], row["mapX"]],
        popup=folium.Popup(popup_html, max_width=350),
        tooltip=row["facltNm"]
    ).add_to(cluster)

m.save("go_camping02.html")


전체 캠핑장 숫자 :  3119


,contentId,facltNm,lineIntro,intro,allar,insrncAt,trsagntNo,bizrno,facltDivNm,mangeDivNm,...,frprvtWrppCo,frprvtSandCo,fireSensorCo,themaEnvrnCl,eqpmnLendCl,animalCmgCl,tourEraCl,firstImageUrl,createdtime,modifiedtime
0,100242,라라플레이스&글램핑,하조대 해변에서 보도거리 1분 위치에 있는 글램핑장,라라플레이스는 2022년 오픈한 글램핑장이다. 글램핑장 안에 온수를 자유롭게 이용할...,1648,Y,2021000026,NaN,민간,직영,...,0,0,12,"일출명소,액티비티,여름물놀이",NaN,가능,NaN,https://gocamping.or.kr/upload/camp/100242/thu...,2022-01-04,2026-11-06
1,102074,공룡자연 팜핑장,치유와 힐링을 할 수 있는 체험캠핑장,공룡자연농원에서의 힐링과 휴식/ 자연과 가족의 소통,594,Y,2026000001,306-34-36279,민간,직영,...,0,0,0,"봄꽃여행,여름물놀이,가을단풍명소,걷기길","텐트,릴선,화로대,난방기구,식기,침낭,일산화탄소감지기",불가능,"봄,여름,가을,겨울",NaN,2026-01-05,2026-11-06
2,1682,섬진강 향가오토캠핑장,섬진강변에서 낭만 캠핑을 즐길 수 있는 향가오토캠핑장,"섬진강 향가오토캠핑장은 10여동의 글램핑과 9개의 방갈로 30여개의 1, 2야영장으...",0,Y,2015000001,787-12-02485,공립,직영,...,0,0,0,NaN,NaN,가능(소형견),NaN,https://gocamping.or.kr/upload/camp/1682/thumb...,2018-09-21,2026-11-05
3,7805,죠스 단체 바베큐장&수영장,탑동계곡에 위치한 레저&글램핑장,죠스글램핑 왕방산 탑동계곡에 위치하고 있다. 탐돌터널 아래 탑동마을회관 건너편이다....,0,Y,2019000002,528-07-01132,민간,직영,...,0,0,0,"액티비티,여름물놀이,가을단풍명소",NaN,가능,NaN,https://gocamping.or.kr/upload/camp/7805/thumb...,2020-01-06,2026-11-05
4,100211,카라반파크 아마존,NaN,"카라반파크 아마존은 완주에 있는 4,443㎡의 대형 카라반 캠핑장입니다. 총 25...",4443,Y,2021000005,134-85-01,민간,직영,...,0,0,25,"액티비티,여름물놀이,가을단풍명소,겨울눈꽃명소,걷기길","난방기구,식기",불가능,"봄,여름,가을,겨울",https://gocamping.or.kr/upload/camp/100211/thu...,2021-12-30,2026-11-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3114,7883,군위전투전승기념공원 캠핑장,NaN,NaN,35248,Y,제2020-000002호,383-19-02152,지자체,위탁,...,0,0,15,걷기길,"릴선,화로대,난방기구,침낭",가능,NaN,https://gocamping.or.kr/upload/camp/7883/thumb...,2020-06-09,2023-11-28
3115,101167,앵봉산 가족캠핑장,서울 서북권 최초의 테라스형 캠핌장,은평구의 풍부한 자연환경을 활용하여 서북권 최초의 테라스형 앵봉산 가족캠핑장을 조성,12521,Y,2023-2,111-83-00088,지자체,위탁,...,1,3,3,"가을단풍명소,걷기길",NaN,불가능,"봄,여름,가을,겨울",https://gocamping.or.kr/upload/camp/101167/thu...,2023-11-02,2023-11-28
3116,418,굼벵이캠핑장,꾸지나무골 해수욕장에 위치한 캠핑장,굼벵이캠핑장은 충남 태안군 이원면에 자리 잡고 있다. 태안군청을 기점으로 약 28k...,4900,Y,NaN,706-08-00123,민간,직영,...,1,1,0,수상레저,NaN,가능(소형견),"봄,여름,가을,겨울",https://gocamping.or.kr/upload/camp/418/thumb/...,2018-09-21,2023-11-20
3117,826,도래재별빛마을캠핑장,밤하늘의 별빛을 감상하며 즐기는 낭만 캠핑,도래재별빛마을 캠핑장은 꼬불꼬불 산길을 돌고 돌아 도래재터널 입구에 도착하면 밀양시...,0,Y,NaN,562-18-0158,민간,직영,...,0,0,0,"일출명소,일몰명소,봄꽃여행,여름물놀이,가을단풍명소,겨울눈꽃명소,걷기길",NaN,불가능,NaN,https://gocamping.or.kr/upload/camp/826/thumb/...,2018-09-21,2023-11-16


In [12]:

import pandas as pd
import folium
from folium.plugins import MarkerCluster
from branca.element import Element
import json

df = pd.read_excel("go_camping.xlsx")
print("전체 캠핑장 숫자 : ", len(df))
display(df)

df["mapX"] = pd.to_numeric(df["mapX"], errors="coerce")
df["mapY"] = pd.to_numeric(df["mapY"], errors="coerce")
df = df.dropna(subset=["mapX", "mapY"])

m = folium.Map(location=[36.5, 127.8], zoom_start=7)
cluster = MarkerCluster().add_to(m)

search_data = []

for _, row in df.iterrows():

    resve_url = row["resveUrl"]
    reserve_html = ""

    if pd.notna(resve_url) and str(resve_url).strip():
        reserve_html = f"""
        <a href="{resve_url}" target="_blank"
        style="background-color:blue;display:block;
        border-radius:10px;color:#fff;padding:10px 0;
        text-align:center;text-decoration:none;">
            예약하기
        </a>
        """

    popup_html = f"""
    <div style="width:300px;">
        <img src="{row['firstImageUrl']}"
        style="width:100%;height:150px;object-fit:cover;
        border-radius:10px">

        <h3>{row["facltNm"]}</h3>
        <p>주소: {row["addr1"]}</p>
        <p>전화번호: {row["tel"]}</p>
        {reserve_html}
    </div>
    """

    marker = folium.Marker(
        location=[row["mapY"], row["mapX"]],
        popup=folium.Popup(popup_html, max_width=350),
        tooltip=row["facltNm"]
    ).add_to(cluster)

    search_data.append({
        "name": str(row["facltNm"]),
        "address": str(row["addr1"]),
        "lat": float(row["mapY"]),
        "lng": float(row["mapX"]),
        "marker": marker.get_name()
    })

# 검색창 HTML + CSS + JavaScript
search_json = json.dumps(search_data, ensure_ascii=False)

search_html = """
<div id="camp-search">
    <input id="camp-input" type="text"
           placeholder="캠핑장 이름을 검색하세요">
    <div id="camp-results"></div>
</div>

<style>
#camp-search {
    position: fixed;
    top: 15px;
    left: 50%;
    transform: translateX(-50%);
    z-index: 9999;
    width: 360px;
    max-width: 90%;
    font-family: sans-serif;
}

#camp-input {
    width: 100%;
    box-sizing: border-box;
    padding: 14px 18px;
    border: 1px solid #ddd;
    border-radius: 10px;
    font-size: 15px;
    background: white;
    box-shadow: 0 3px 12px #0002;
    outline: none;
}

#camp-results {
    margin-top: 5px;
    max-height: 300px;
    overflow-y: auto;
    background: white;
    border-radius: 10px;
    box-shadow: 0 3px 12px #0002;
    display: none;
}

.camp-result {
    padding: 10px 15px;
    cursor: pointer;
    border-bottom: 1px solid #eee;
}

.camp-result:hover {
    background: #f0f5ff;
}

.camp-result small {
    display: block;
    color: #777;
    margin-top: 3px;
}
</style>

<script>
document.addEventListener("DOMContentLoaded", function() {

    const data = __SEARCH_DATA__;
    const map = __MAP_NAME__;
    const cluster = __CLUSTER_NAME__;
    const input = document.getElementById("camp-input");
    const results = document.getElementById("camp-results");

    input.addEventListener("input", function() {
        const keyword = input.value.trim().toLowerCase();
        results.innerHTML = "";

        if (!keyword) {
            results.style.display = "none";
            return;
        }

        const filtered = data.filter(item =>
            item.name.toLowerCase().includes(keyword)
        ).slice(0, 20);

        results.style.display = "block";

        if (filtered.length === 0) {
            results.textContent = "검색 결과가 없습니다.";
            return;
        }

        filtered.forEach(item => {
            const div = document.createElement("div");
            div.className = "camp-result";

            const title = document.createElement("strong");
            title.textContent = item.name;

            const address = document.createElement("small");
            address.textContent = item.address;

            div.appendChild(title);
            div.appendChild(address);

            div.addEventListener("click", function() {
                const marker = window[item.marker];

                input.value = item.name;
                results.style.display = "none";

                cluster.zoomToShowLayer(marker, function() {
                    map.setView([item.lat, item.lng],
                                Math.max(map.getZoom(), 14));
                    marker.openPopup();
                });
            });

            results.appendChild(div);
        });
    });
});
</script>
"""

search_html = search_html.replace(
    "__SEARCH_DATA__", search_json
).replace(
    "__MAP_NAME__", m.get_name()
).replace(
    "__CLUSTER_NAME__", cluster.get_name()
)

m.get_root().html.add_child(Element(search_html))

m.save("index.html")


전체 캠핑장 숫자 :  3119


,contentId,facltNm,lineIntro,intro,allar,insrncAt,trsagntNo,bizrno,facltDivNm,mangeDivNm,...,frprvtWrppCo,frprvtSandCo,fireSensorCo,themaEnvrnCl,eqpmnLendCl,animalCmgCl,tourEraCl,firstImageUrl,createdtime,modifiedtime
0,100242,라라플레이스&글램핑,하조대 해변에서 보도거리 1분 위치에 있는 글램핑장,라라플레이스는 2022년 오픈한 글램핑장이다. 글램핑장 안에 온수를 자유롭게 이용할...,1648,Y,2021000026,NaN,민간,직영,...,0,0,12,"일출명소,액티비티,여름물놀이",NaN,가능,NaN,https://gocamping.or.kr/upload/camp/100242/thu...,2022-01-04,2026-11-06
1,102074,공룡자연 팜핑장,치유와 힐링을 할 수 있는 체험캠핑장,공룡자연농원에서의 힐링과 휴식/ 자연과 가족의 소통,594,Y,2026000001,306-34-36279,민간,직영,...,0,0,0,"봄꽃여행,여름물놀이,가을단풍명소,걷기길","텐트,릴선,화로대,난방기구,식기,침낭,일산화탄소감지기",불가능,"봄,여름,가을,겨울",NaN,2026-01-05,2026-11-06
2,1682,섬진강 향가오토캠핑장,섬진강변에서 낭만 캠핑을 즐길 수 있는 향가오토캠핑장,"섬진강 향가오토캠핑장은 10여동의 글램핑과 9개의 방갈로 30여개의 1, 2야영장으...",0,Y,2015000001,787-12-02485,공립,직영,...,0,0,0,NaN,NaN,가능(소형견),NaN,https://gocamping.or.kr/upload/camp/1682/thumb...,2018-09-21,2026-11-05
3,7805,죠스 단체 바베큐장&수영장,탑동계곡에 위치한 레저&글램핑장,죠스글램핑 왕방산 탑동계곡에 위치하고 있다. 탐돌터널 아래 탑동마을회관 건너편이다....,0,Y,2019000002,528-07-01132,민간,직영,...,0,0,0,"액티비티,여름물놀이,가을단풍명소",NaN,가능,NaN,https://gocamping.or.kr/upload/camp/7805/thumb...,2020-01-06,2026-11-05
4,100211,카라반파크 아마존,NaN,"카라반파크 아마존은 완주에 있는 4,443㎡의 대형 카라반 캠핑장입니다. 총 25...",4443,Y,2021000005,134-85-01,민간,직영,...,0,0,25,"액티비티,여름물놀이,가을단풍명소,겨울눈꽃명소,걷기길","난방기구,식기",불가능,"봄,여름,가을,겨울",https://gocamping.or.kr/upload/camp/100211/thu...,2021-12-30,2026-11-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3114,7883,군위전투전승기념공원 캠핑장,NaN,NaN,35248,Y,제2020-000002호,383-19-02152,지자체,위탁,...,0,0,15,걷기길,"릴선,화로대,난방기구,침낭",가능,NaN,https://gocamping.or.kr/upload/camp/7883/thumb...,2020-06-09,2023-11-28
3115,101167,앵봉산 가족캠핑장,서울 서북권 최초의 테라스형 캠핌장,은평구의 풍부한 자연환경을 활용하여 서북권 최초의 테라스형 앵봉산 가족캠핑장을 조성,12521,Y,2023-2,111-83-00088,지자체,위탁,...,1,3,3,"가을단풍명소,걷기길",NaN,불가능,"봄,여름,가을,겨울",https://gocamping.or.kr/upload/camp/101167/thu...,2023-11-02,2023-11-28
3116,418,굼벵이캠핑장,꾸지나무골 해수욕장에 위치한 캠핑장,굼벵이캠핑장은 충남 태안군 이원면에 자리 잡고 있다. 태안군청을 기점으로 약 28k...,4900,Y,NaN,706-08-00123,민간,직영,...,1,1,0,수상레저,NaN,가능(소형견),"봄,여름,가을,겨울",https://gocamping.or.kr/upload/camp/418/thumb/...,2018-09-21,2023-11-20
3117,826,도래재별빛마을캠핑장,밤하늘의 별빛을 감상하며 즐기는 낭만 캠핑,도래재별빛마을 캠핑장은 꼬불꼬불 산길을 돌고 돌아 도래재터널 입구에 도착하면 밀양시...,0,Y,NaN,562-18-0158,민간,직영,...,0,0,0,"일출명소,일몰명소,봄꽃여행,여름물놀이,가을단풍명소,겨울눈꽃명소,걷기길",NaN,불가능,NaN,https://gocamping.or.kr/upload/camp/826/thumb/...,2018-09-21,2023-11-16
